In [ ]:
from transformers import ViTForImageClassification, ViTFeatureExtractor, Trainer, TrainingArguments
from PIL import Image
import torch
import pandas as pd
import os
from torch.utils.data import Dataset, random_split

In [ ]:
model_name = "./models/models--google--vit-base-patch16-224/snapshots/3f49326eb077187dfe1c2a2bb15fbd74e6ab91e3"
model = ViTForImageClassification.from_pretrained(model_name, num_labels=2,ignore_mismatched_sizes=True)
feature_extractor = ViTFeatureExtractor.from_pretrained(model_name)

class HouseDataset(Dataset):
    def __init__(self, df, image_folder, feature_extractor):
        self.df = df
        self.image_folder = image_folder
        self.feature_extractor = feature_extractor

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        image_path = os.path.join(self.image_folder, self.df.iloc[idx]["image_name"])
        image = Image.open(image_path).convert("RGB")
        inputs = self.feature_extractor(images=image, return_tensors="pt")
        label = torch.tensor(self.df.iloc[idx]["class"], dtype=torch.long)
        return {"pixel_values": inputs["pixel_values"].squeeze(), "labels": label}

Some weights of ViTForImageClassification were not initialized from the model checkpoint at ./models/models--google--vit-base-patch16-224/snapshots/3f49326eb077187dfe1c2a2bb15fbd74e6ab91e3 and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([1000]) in the checkpoint and torch.Size([2]) in the model instantiated
- classifier.weight: found shape torch.Size([1000, 768]) in the checkpoint and torch.Size([2, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/home/ai5039/.conda/envs/imagetask/lib/python3.13/site-packages/transformers/models/vit/feature_extraction_vit.py:30: FutureWarning: The class ViTFeatureExtractor is deprecated and will be removed in version 5 of Transformers. Please use ViTImageProcessor instead.
  warnings.warn(


In [ ]:
# for param in model.parameters():
#     param.requires_grad = True

# for param in model.vit.encoder.layer[:-7].parameters():
#     param.requires_grad = True

In [ ]:
df = pd.read_csv("./data/train.csv")
image_folder = "./data/train/train"
dataset = HouseDataset(df, image_folder, feature_extractor)

In [ ]:
train_dataset, eval_dataset = random_split(dataset, [0.8, 0.2])

In [ ]:
from sklearn.metrics import accuracy_score, f1_score
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    # If predictions are logits, take argmax to get class labels
    predictions = predictions.argmax(-1)
    accuracy = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average="weighted")
    return {
        "accuracy": accuracy,
        "f1": f1,
    }

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=256,
    per_device_eval_batch_size=256,
    num_train_epochs=25,
    weight_decay=0.01,
    logging_steps=10,
    report_to="none",
    fp16 = True,
    dataloader_num_workers = 4
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,  # Add evaluation dataset
    compute_metrics = compute_metrics
)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.500500,0.336315,0.869492,0.869552
2,0.229700,0.235933,0.901695,0.901724
3,0.129400,0.208620,0.913559,0.913360
4,0.081700,0.166330,0.933898,0.933914
5,0.043300,0.161808,0.935593,0.935568
6,0.027400,0.178619,0.938983,0.938806
7,0.015400,0.163826,0.935593,0.935613
8,0.010400,0.175360,0.945763,0.945715
9,0.007300,0.177164,0.942373,0.942350
10,0.005600,0.182053,0.944068,0.944026


TrainOutput(global_step=250, training_loss=0.04366607688367367, metrics={'train_runtime': 480.5304, 'train_samples_per_second': 122.937, 'train_steps_per_second': 0.52, 'total_flos': 4.5778392864820224e+18, 'train_loss': 0.04366607688367367, 'epoch': 25.0})

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
def predict(images_folder, model, feature_extractor, device):
    model.eval()
    predictions = []
    image_files = [f for f in os.listdir(images_folder) if f.endswith(".jpg") or f.endswith(".png")]

    for image_file in image_files:
        image_path = os.path.join(images_folder, image_file)
        image = Image.open(image_path).convert("RGB")
        inputs = feature_extractor(images=image, return_tensors="pt").to(device)
        with torch.no_grad():
            outputs = model(**inputs)
            predicted_label = torch.argmax(outputs.logits, dim=-1).item()
        predictions.append((image_file, predicted_label))

    return predictions

In [ ]:
test_folder = "./data/test/test"
predictions = predict(test_folder, model, feature_extractor,device)

In [ ]:
submission = pd.read_csv("./data/sample_submission.csv")

In [ ]:
submission = pd.DataFrame(predictions)

In [ ]:
submission = submission.rename(columns={0: "id", 1: "answer"})
submission["id"] = submission["id"].str.replace(".jpg", "")

In [ ]:
submission

,id,answer
0,2efc53fd,0
1,0806349e,1
2,e0e92023,0
3,4d1b8a1e,0
4,9f60d126,0
...,...,...
1545,10b8ec65,1
1546,180ce114,1
1547,f1e17810,1
1548,fa5946bc,0


In [ ]:
submission.to_csv("./submission/ViTsubmission.csv",index=False)

In [ ]:
#kaggle competitions download -c image-processing-house-recognition

In [ ]:
#kaggle competitions submit -c image-processing-house-recognition -f ./submission/ViTsubmission.csv -m "ViTBase baseline"